In [2]:
# =========================
# Latent intersection experiment (NO recompute of activations)
# Score = HR_freq - BC_freq
# - Rankings per dataset (4)
# - Rankings per split (13)
# - Intersection counts over top-N membership
# =========================

import json
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

# -----------------------
# CONFIG (edit here)
# -----------------------
DTYPE = torch.float16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAE config
SAE_BASE_DIR = Path("../exp1_v3/saes")  # <- change if needed
SAE_WIDTH = "16k"
SAE_L0 = "14"
SAE_LAYERS = [9, 20, 31]          # Gemma-Scope convention
BLOCK_IDXS = [l + 1 for l in SAE_LAYERS]  # [10, 21, 32]
POS = -2                          # already baked into cached activations you computed

# Intersection definition
INTERSECTION_TOP_N = 1000         # <- IMPORTANT: intersection is computed over top-N membership

# Generation JSONs
GEN_COCONOT = Path("generations/coconot/original/evaluated/train_generations_wildguard.json")
GEN_SORRY   = Path("generations/sorrybench/evaluated/sorrybench_base440_generations_wildguard.json")
GEN_WG      = Path("generations/original_wildguard/evaluated/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_wildguard.json")
GEN_XS      = Path("generations/xstest/evaluated/test_generations_wildguard.json")

# Activation caches (NO recompute)
ACTS_COCONOT = Path("activations/coconot/original/train_l10_21_32_pos-2_residpre.pt")
ACTS_SORRY   = Path("activations/sorrybench_base440_l10_21_32_pos-2_residpre.pt")
ACTS_WG      = Path("activations/recomputed/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_l10_21_32_pos-2_residpre.pt")  # <-- USE THIS
ACTS_XS      = Path("activations/xstest/test_l10_21_32_pos-2_residpre.pt")

# -----------------------
# Helpers: SAE + loading
# -----------------------
class JumpReLUSAE(nn.Module):
    def __init__(self, d_model, d_sae):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.threshold = nn.Parameter(torch.zeros(d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        mask = pre > self.threshold
        return mask * torch.relu(pre)

def load_local_sae(layer: int, device=DEVICE, dtype=DTYPE):
    params_path = (
        SAE_BASE_DIR
        / f"layer_{layer}"
        / f"width_{SAE_WIDTH}"
        / f"average_l0_{SAE_L0}"
        / "params.npz"
    )
    params = np.load(params_path)
    sae = JumpReLUSAE(params["W_enc"].shape[0], params["W_enc"].shape[1]).to(device, dtype)
    sae.W_enc.data.copy_(torch.from_numpy(params["W_enc"]).to(device, dtype))
    sae.W_dec.data.copy_(torch.from_numpy(params["W_dec"]).to(device, dtype))
    sae.b_enc.data.copy_(torch.from_numpy(params["b_enc"]).to(device, dtype))
    sae.b_dec.data.copy_(torch.from_numpy(params["b_dec"]).to(device, dtype))
    sae.threshold.data.copy_(torch.from_numpy(params["threshold"]).to(device, dtype))
    sae.eval()
    return sae

def parse_binary_is_refusal(val):
    """Return 0/1 or None (drops 'n/a', None, other junk)."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, (int, np.integer)):
        return int(val) if val in (0, 1) else None
    if isinstance(val, float):
        return int(val) if val in (0.0, 1.0) else None
    if isinstance(val, str):
        s = val.strip().lower()
        if s in ("0", "1"):
            return int(s)
        return None
    return None

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def load_acts(path: Path):
    obj = torch.load(path, map_location="cpu")
    # minimal support: either dict-with-acts or raw tensor
    if isinstance(obj, dict) and "acts" in obj:
        acts = obj["acts"]
        meta = {k: v for k, v in obj.items() if k != "acts"}
        return acts, meta
    if torch.is_tensor(obj):
        return obj, {}
    raise ValueError(f"Activation file not understood: {path}")

def align_records_and_acts(records, acts):
    """
    Drops non-binary is_refusal rows from records,
    and drops the same rows from acts (assumes acts aligned to original record order).
    """
    keep = []
    y = []
    for i, r in enumerate(records):
        v = parse_binary_is_refusal(r.get("is_refusal", None))
        if v is None:
            continue
        keep.append(i)
        y.append(v)

    y = torch.tensor(y, dtype=torch.long)
    keep = torch.tensor(keep, dtype=torch.long)

    # acts can be [N, D] or [N, L, D]
    if acts.size(0) == len(records):
        acts_kept = acts.index_select(0, keep)
    elif acts.size(0) == keep.numel():
        # already filtered upstream (rare) — just trust it
        acts_kept = acts
    else:
        raise ValueError(f"Length mismatch: records={len(records)} but acts[0]={acts.size(0)} for this dataset")

    recs_kept = [records[i] for i in keep.tolist()]
    return recs_kept, y, acts_kept

def select_layer_slice(acts, layer: int, meta: dict):
    """
    Returns [N, D] for a given SAE layer from multi-layer cache [N, L, D].
    Assumes ordering matches BLOCK_IDXS = [10,21,32].
    """
    if acts.dim() == 2:
        return acts
    assert acts.dim() == 3, f"Expected acts [N,L,D] or [N,D], got {tuple(acts.shape)}"

    # if metadata provides block_idxs, use it; otherwise assume [10,21,32]
    block_idxs = meta.get("block_idxs", BLOCK_IDXS)
    if isinstance(block_idxs, torch.Tensor):
        block_idxs = block_idxs.tolist()

    wanted_block = layer + 1
    if wanted_block not in block_idxs:
        raise ValueError(f"wanted block {wanted_block} not in block_idxs={block_idxs}")
    col = block_idxs.index(wanted_block)
    return acts[:, col, :]

# -----------------------
# Load + align all datasets
# -----------------------
cn_records_raw = load_json(GEN_COCONOT)
sb_records_raw = load_json(GEN_SORRY)
wg_records_raw = load_json(GEN_WG)
xs_records_raw = load_json(GEN_XS)

cn_acts_raw, cn_meta = load_acts(ACTS_COCONOT)
sb_acts_raw, sb_meta = load_acts(ACTS_SORRY)
wg_acts_raw, wg_meta = load_acts(ACTS_WG)      # <-- correct 3-layer cache
xs_acts_raw, xs_meta = load_acts(ACTS_XS)

cn_records, cn_isref, cn_acts = align_records_and_acts(cn_records_raw, cn_acts_raw)
sb_records, sb_isref, sb_acts = align_records_and_acts(sb_records_raw, sb_acts_raw)
wg_records, wg_isref, wg_acts = align_records_and_acts(wg_records_raw, wg_acts_raw)
xs_records, xs_isref, xs_acts = align_records_and_acts(xs_records_raw, xs_acts_raw)

print("After dropping non-binary is_refusal rows:")
print(f"  coconot:    {len(cn_records)}")
print(f"  sorrybench: {len(sb_records)}")
print(f"  wildguard:  {len(wg_records)}")
print(f"  xstest:     {len(xs_records)}")

# -----------------------
# Define HR / BC indices (as per your spec)
#   - HR/BC is refusal/compliance from is_refusal
#   - harmful/benign comes from dataset label where present
#   - coconot & sorrybench: harmful-only -> BC comes from WildGuard benign+compliant
# -----------------------
def idx_wg_benign_compliant(records, isref):
    # benign/unharmful interchangeable: prompt_harm_label == "unharmful"
    m = []
    for i, r in enumerate(records):
        if (r.get("prompt_harm_label") == "unharmful") and (int(isref[i]) == 0):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

def idx_wg_harmful_refusal(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("prompt_harm_label") == "harmful") and (int(isref[i]) == 1):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

def idx_xs_benign_compliant(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("label") == "safe") and (int(isref[i]) == 0):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

def idx_xs_harmful_refusal(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("label") == "unsafe") and (int(isref[i]) == 1):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

def idx_all_refusals(isref):
    return (isref == 1).nonzero(as_tuple=True)[0]

# Shared BC pool for coconot/sorrybench (WildGuard benign compliance)
WG_BC_IDX = idx_wg_benign_compliant(wg_records, wg_isref)

# Dataset-level HR/BC
dataset_specs = {
    "coconot_all":    {"hr_idx": idx_all_refusals(cn_isref), "bc_src": ("wildguard", WG_BC_IDX)},
    "sorrybench_all": {"hr_idx": idx_all_refusals(sb_isref), "bc_src": ("wildguard", WG_BC_IDX)},
    "wildguard_all":  {"hr_idx": idx_wg_harmful_refusal(wg_records, wg_isref), "bc_src": ("wildguard", WG_BC_IDX)},
    "xstest_all":     {"hr_idx": idx_xs_harmful_refusal(xs_records, xs_isref), "bc_src": ("xstest", idx_xs_benign_compliant(xs_records, xs_isref))},
}

# -----------------------
# Split definitions (13) — exactly your list
# -----------------------
coconot_split_specs = [
    ("coconot_all", None),
    ("coconot_cat_humanizing_requests", "Humanizing requests"),
    ("coconot_cat_incomplete_requests", "Incomplete requests"),
    ("coconot_cat_indeterminate_requests", "Indeterminate requests"),
    ("coconot_cat_requests_with_safety_concerns", "Requests with safety concerns"),
    ("coconot_cat_unsupported_requests", "Unsupported requests"),
]

def coconot_hr_idx_for_category(records, isref, category_value):
    m = []
    for i, r in enumerate(records):
        if category_value is not None and r.get("category") != category_value:
            continue
        if int(isref[i]) == 1:
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

# SorryBench numeric category mapping from your image:
#   1-5   -> hate_speech_generation
#   6-24  -> assistance_with_crimes_or_torts
#   25-39 -> potentially_inappropriate_topics
#   40-44 -> potentially_unqualified_advice
def sorrybench_bucket(cat_num: int):
    if 1 <= cat_num <= 5:   return "sorrybench_hate_speech_generation"
    if 6 <= cat_num <= 24:  return "sorrybench_assistance_with_crimes_or_torts"
    if 25 <= cat_num <= 39: return "sorrybench_potentially_inappropriate_topics"
    if 40 <= cat_num <= 44: return "sorrybench_potentially_unqualified_advice"
    return None

sorrybench_split_names = [
    "sorrybench_all",
    "sorrybench_hate_speech_generation",
    "sorrybench_assistance_with_crimes_or_torts",
    "sorrybench_potentially_inappropriate_topics",
    "sorrybench_potentially_unqualified_advice",
]

def sorrybench_hr_idx_for_bucket(records, isref, bucket_name):
    m = []
    for i, r in enumerate(records):
        if int(isref[i]) != 1:
            continue
        if bucket_name == "sorrybench_all":
            m.append(i)
            continue
        c = r.get("category", None)
        try:
            c = int(c)
        except Exception:
            continue
        b = sorrybench_bucket(c)
        if b == bucket_name:
            m.append(i)
    return torch.tensor(m, dtype=torch.long)

# Build split_specs dict: split_name -> (hr_acts_src, hr_idx, bc_acts_src, bc_idx)
split_specs = {}

# coconot splits (BC always WildGuard benign compliance)
for split_name, cat_value in coconot_split_specs:
    split_specs[split_name] = ("coconot", coconot_hr_idx_for_category(cn_records, cn_isref, cat_value),
                               "wildguard", WG_BC_IDX)

# sorrybench splits (BC always WildGuard benign compliance)
for split_name in sorrybench_split_names:
    split_specs[split_name] = ("sorrybench", sorrybench_hr_idx_for_bucket(sb_records, sb_isref, split_name),
                               "wildguard", WG_BC_IDX)

# wildguard/xstest splits
split_specs["wildguard_all"] = ("wildguard", idx_wg_harmful_refusal(wg_records, wg_isref),
                                "wildguard", WG_BC_IDX)
split_specs["xstest_all"]    = ("xstest", idx_xs_harmful_refusal(xs_records, xs_isref),
                                "xstest", idx_xs_benign_compliant(xs_records, xs_isref))

# Sanity: ensure we have exactly the 13 splits you listed
expected_splits = [
    "coconot_all",
    "coconot_cat_humanizing_requests",
    "coconot_cat_incomplete_requests",
    "coconot_cat_indeterminate_requests",
    "coconot_cat_requests_with_safety_concerns",
    "coconot_cat_unsupported_requests",
    "sorrybench_all",
    "sorrybench_hate_speech_generation",
    "sorrybench_assistance_with_crimes_or_torts",
    "sorrybench_potentially_inappropriate_topics",
    "sorrybench_potentially_unqualified_advice",
    "wildguard_all",
    "xstest_all",
]
assert set(split_specs.keys()) == set(expected_splits), f"Split key mismatch: {sorted(split_specs.keys())}"

print("\nSplits (13):")
for s in expected_splits:
    print(" -", s)

# -----------------------
# Core: compute score = HR_freq - BC_freq for all latents
# -----------------------
def fraction_active(sae: JumpReLUSAE, acts_2d: torch.Tensor, idx: torch.Tensor, batch=512):
    """
    JumpReLU active fraction per latent over rows selected by idx.
    acts_2d: [N, D] on CPU
    idx: 1D long indices into N
    returns: [code_size] float32 on CPU
    """
    if idx.numel() == 0:
        # no examples -> all zeros (avoid div0)
        return torch.zeros(sae.W_enc.shape[1], dtype=torch.float32)

    code_size = sae.W_enc.shape[1]
    count = torch.zeros(code_size, dtype=torch.float32)

    order = idx
    n = float(order.numel())

    for start in range(0, order.numel(), batch):
        sl = order[start:start+batch]
        x = acts_2d.index_select(0, sl).to(DEVICE, dtype=sae.W_enc.dtype)
        with torch.no_grad():
            z = sae.encode(x)
            active = (z > 0)
        count += active.sum(dim=0).float().cpu()
        del x, z, active
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return count / n

def rank_latents_by_score(score: torch.Tensor):
    # returns python list[int] of latent ids sorted desc by score
    return torch.argsort(score, descending=True).tolist()

# Source->acts+meta bundles for easy routing
ACTS_BY_SRC = {
    "coconot":   (cn_acts, cn_meta),
    "sorrybench":(sb_acts, sb_meta),
    "wildguard": (wg_acts, wg_meta),
    "xstest":    (xs_acts, xs_meta),
}

# Outputs
scores_by_layer_dataset = {L: {} for L in SAE_LAYERS}
ranked_by_layer_dataset = {L: {} for L in SAE_LAYERS}

scores_by_layer_split   = {L: {} for L in SAE_LAYERS}
ranked_by_layer_split   = {L: {} for L in SAE_LAYERS}

intersection_by_layer_dataset = {}
intersection_by_layer_split   = {}

# -----------------------
# Compute everything per layer
# -----------------------
for L in SAE_LAYERS:
    print(f"\n=== LAYER {L} ===")
    sae = load_local_sae(L)

    # materialize per-source [N,D] view for this layer (from [N,3,D])
    acts2d = {}
    for src, (acts, meta) in ACTS_BY_SRC.items():
        acts2d[src] = select_layer_slice(acts, L, meta)  # [N,D] CPU

    # ---- Dataset-level rankings (4)
    for ds_name, spec in dataset_specs.items():
        hr_idx = spec["hr_idx"]
        bc_src, bc_idx = spec["bc_src"]

        # HR always from dataset implied by name
        hr_src = "coconot" if ds_name.startswith("coconot") else \
                 "sorrybench" if ds_name.startswith("sorrybench") else \
                 "wildguard" if ds_name.startswith("wildguard") else \
                 "xstest"

        hr_freq = fraction_active(sae, acts2d[hr_src], hr_idx)
        bc_freq = fraction_active(sae, acts2d[bc_src], bc_idx)
        score = hr_freq - bc_freq

        scores_by_layer_dataset[L][ds_name] = score
        ranked_by_layer_dataset[L][ds_name] = rank_latents_by_score(score)

    # ---- Split-level rankings (13)
    for split_name in expected_splits:
        hr_src, hr_idx, bc_src, bc_idx = split_specs[split_name]
        hr_freq = fraction_active(sae, acts2d[hr_src], hr_idx)
        bc_freq = fraction_active(sae, acts2d[bc_src], bc_idx)
        score = hr_freq - bc_freq

        scores_by_layer_split[L][split_name] = score
        ranked_by_layer_split[L][split_name] = rank_latents_by_score(score)

    # -----------------------
    # Intersections (top-N membership counts)
    # -----------------------
    def topn_membership_counts(rankings: dict, topn: int):
        """
        rankings: name -> list[int] (full ranking)
        returns: list of tuples (latent_id, count, rank_sum) sorted by count desc then rank_sum asc
        and strict_intersection: [latent_id] that appear in topn of ALL lists
        """
        names = list(rankings.keys())
        K = len(names)
        code_size = len(next(iter(rankings.values())))

        # precompute rank positions (for tie-break)
        rank_pos = {name: torch.empty(code_size, dtype=torch.int32) for name in names}
        for name in names:
            r = rankings[name]
            for pos, lid in enumerate(r):
                rank_pos[name][lid] = pos

        counts = torch.zeros(code_size, dtype=torch.int32)
        rsum   = torch.zeros(code_size, dtype=torch.int64)

        for name in names:
            top = rankings[name][:topn]
            top_t = torch.tensor(top, dtype=torch.long)
            counts[top_t] += 1
            rsum[top_t] += rank_pos[name][top_t].to(torch.int64)

        # sort by (-count, +rank_sum)
        # (use python sort for simplicity)
        items = [(i, int(counts[i].item()), int(rsum[i].item())) for i in range(code_size) if counts[i] > 0]
        items.sort(key=lambda x: (-x[1], x[2]))

        strict = [i for (i,c,_) in items if c == K]
        return items, strict

    # dataset intersection (4 lists)
    ds_items, ds_strict = topn_membership_counts(ranked_by_layer_dataset[L], INTERSECTION_TOP_N)
    intersection_by_layer_dataset[L] = {
        "topn": INTERSECTION_TOP_N,
        "items": ds_items,          # [(latent_id, count, rank_sum), ...] for latents that appear in at least one top-N
        "strict_intersection": ds_strict,  # latents in top-N of ALL 4 datasets
    }

    # split intersection (13 lists)
    sp_items, sp_strict = topn_membership_counts(ranked_by_layer_split[L], INTERSECTION_TOP_N)
    intersection_by_layer_split[L] = {
        "topn": INTERSECTION_TOP_N,
        "items": sp_items,
        "strict_intersection": sp_strict,  # latents in top-N of ALL 13 splits
    }

    # cleanup
    del sae
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nDone.\n"
      "Key outputs:\n"
      "- ranked_by_layer_dataset[layer][dataset_name] -> full ranking list of latent ids\n"
      "- ranked_by_layer_split[layer][split_name]     -> full ranking list of latent ids\n"
      "- intersection_by_layer_dataset[layer]['items'] -> (latent_id,count,rank_sum) over top-N\n"
      "- intersection_by_layer_dataset[layer]['strict_intersection'] -> latents in top-N of ALL 4 datasets\n"
      "- intersection_by_layer_split[layer]['strict_intersection']   -> latents in top-N of ALL 11 splits\n")


After dropping non-binary is_refusal rows:
  coconot:    11473
  sorrybench: 440
  wildguard:  915
  xstest:     450

Splits (13):
 - coconot_all
 - coconot_cat_humanizing_requests
 - coconot_cat_incomplete_requests
 - coconot_cat_indeterminate_requests
 - coconot_cat_requests_with_safety_concerns
 - coconot_cat_unsupported_requests
 - sorrybench_all
 - sorrybench_hate_speech_generation
 - sorrybench_assistance_with_crimes_or_torts
 - sorrybench_potentially_inappropriate_topics
 - sorrybench_potentially_unqualified_advice
 - wildguard_all
 - xstest_all

=== LAYER 9 ===

=== LAYER 20 ===

=== LAYER 31 ===

Done.
Key outputs:
- ranked_by_layer_dataset[layer][dataset_name] -> full ranking list of latent ids
- ranked_by_layer_split[layer][split_name]     -> full ranking list of latent ids
- intersection_by_layer_dataset[layer]['items'] -> (latent_id,count,rank_sum) over top-N
- intersection_by_layer_dataset[layer]['strict_intersection'] -> latents in top-N of ALL 4 datasets
- intersection_

In [3]:
# =========================
# DISPLAY RESULTS (Jupyter)
# =========================
import pandas as pd
from IPython.display import display, Markdown

SHOW_TOP = 50                 # how many rows to display per list
SHOW_INTERSECTION_TOP = 200   # how many rows of intersection table to display
TOPN_FOR_INTERSECTION = INTERSECTION_TOP_N  # just echo back what you used

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

def _df_from_score(score_tensor: torch.Tensor):
    # score_tensor: [code_size] on CPU
    s = score_tensor.detach().cpu().float().numpy()
    df = pd.DataFrame({"latent_id": np.arange(len(s), dtype=np.int32), "score": s})
    df = df.sort_values("score", ascending=False, kind="mergesort").reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1, dtype=np.int32)
    return df[["rank", "latent_id", "score"]]

def _display_grouped_top(df_all, group_col, title):
    display(Markdown(f"### {title}"))
    # show top SHOW_TOP per group
    out = df_all.groupby(group_col, group_keys=False).head(SHOW_TOP)
    display(out)

def _intersection_items_to_df(items):
    # items: list[(latent_id, count, rank_sum)]
    return (
        pd.DataFrame(items, columns=["latent_id", "count_in_topN_lists", "rank_sum_tiebreak"])
        .sort_values(["count_in_topN_lists", "rank_sum_tiebreak"], ascending=[False, True])
        .reset_index(drop=True)
    )

# Dataset-level order (unchanged – still 4 big pools)
DATASET_ORDER = ["coconot_all", "sorrybench_all", "wildguard_all", "xstest_all"]

# --- CHANGED: use expected_splits from above (which no longer includes coconot_all / sorrybench_all)
SPLIT_ORDER = expected_splits
NUM_SPLITS = len(SPLIT_ORDER)   # will be 11 now

display(Markdown(f"# Latent rankings + intersections (score = HR_freq − BC_freq)"))
display(Markdown(f"- Intersection is computed over **top {TOPN_FOR_INTERSECTION}** membership."))

for L in SAE_LAYERS:
    display(Markdown(f"## Layer {L}"))

    # -----------------------
    # DATASET TABLES (4)
    # -----------------------
    ds_rows = []
    for ds_name in DATASET_ORDER:
        df = _df_from_score(scores_by_layer_dataset[L][ds_name])
        df.insert(0, "dataset", ds_name)
        ds_rows.append(df)
    df_ds_all = pd.concat(ds_rows, ignore_index=True)

    _display_grouped_top(
        df_ds_all,
        "dataset",
        f"Per-dataset latent rankings (top {SHOW_TOP} each)",
    )

    # -----------------------
    # SPLIT TABLES (now 11)
    # -----------------------
    sp_rows = []
    for split_name in SPLIT_ORDER:
        df = _df_from_score(scores_by_layer_split[L][split_name])
        df.insert(0, "split", split_name)
        sp_rows.append(df)
    df_sp_all = pd.concat(sp_rows, ignore_index=True)

    _display_grouped_top(
        df_sp_all,
        "split",
        f"Per-split latent rankings (top {SHOW_TOP} each)",
    )

    # -----------------------
    # INTERSECTIONS (datasets)
    # -----------------------
    display(Markdown(f"### Dataset intersection over top {TOPN_FOR_INTERSECTION}"))
    ds_items = intersection_by_layer_dataset[L]["items"]
    ds_strict = intersection_by_layer_dataset[L]["strict_intersection"]
    df_int_ds = _intersection_items_to_df(ds_items)

    display(df_int_ds.head(SHOW_INTERSECTION_TOP))
    display(
        Markdown(
            f"**Strict intersection (in top {TOPN_FOR_INTERSECTION} of ALL {len(DATASET_ORDER)} datasets):** "
            f"{len(ds_strict)} latents"
        )
    )
    display(ds_strict[:200])  # show first 200 ids (full list still in ds_strict)

    # -----------------------
    # INTERSECTIONS (splits)
    # -----------------------
    display(Markdown(f"### Split intersection over top {TOPN_FOR_INTERSECTION}"))
    sp_items = intersection_by_layer_split[L]["items"]
    sp_strict = intersection_by_layer_split[L]["strict_intersection"]
    df_int_sp = _intersection_items_to_df(sp_items)

    display(df_int_sp.head(SHOW_INTERSECTION_TOP))
    display(
        Markdown(
            f"**Strict intersection (in top {TOPN_FOR_INTERSECTION} of ALL {NUM_SPLITS} splits):** "
            f"{len(sp_strict)} latents"
        )
    )
    display(sp_strict[:200])  # show first 200 ids (full list still in sp_strict)

display(Markdown("✅ Done displaying."))


# Latent rankings + intersections (score = HR_freq − BC_freq)

- Intersection is computed over **top 1000** membership.

## Layer 9

### Per-dataset latent rankings (top 50 each)

,dataset,rank,latent_id,score
0,coconot_all,1,12891,0.221580
1,coconot_all,2,1754,0.198590
2,coconot_all,3,4501,0.171059
3,coconot_all,4,8001,0.116556
4,coconot_all,5,3894,0.112473
...,...,...,...,...
49197,xstest_all,46,7,0.000000
49198,xstest_all,47,8,0.000000
49199,xstest_all,48,9,0.000000
49200,xstest_all,49,10,0.000000


...

### Split intersection over top 1000

,latent_id,count_in_topN_lists,rank_sum_tiebreak
0,6768,13,79
1,550,13,86
2,1779,13,94
3,14020,13,269
4,10899,13,1994
...,...,...,...
195,10976,13,6213
196,11081,13,6213
197,11002,13,6227
198,11009,13,6233


**Strict intersection (in top 1000 of ALL 13 splits):** 421 latents

[6768,
 550,
 1779,
 14020,
 10899,
 10893,
 10892,
 10926,
 10889,
 10887,
 10886,
 10890,
 10883,
 10894,
 10885,
 10888,
 10895,
 10884,
 10880,
 10882,
 10891,
 10897,
 10879,
 10881,
 10896,
 10931,
 10878,
 10943,
 10940,
 10941,
 10942,
 10944,
 10939,
 10917,
 10876,
 10918,
 10938,
 10934,
 10937,
 10874,
 10916,
 10875,
 10915,
 10919,
 10913,
 10877,
 10898,
 10920,
 10921,
 10929,
 10909,
 10925,
 10922,
 10927,
 10930,
 10936,
 10928,
 10935,
 10923,
 10912,
 10924,
 10901,
 10933,
 10914,
 10903,
 10911,
 10932,
 10947,
 10902,
 10948,
 10949,
 10908,
 10946,
 10873,
 10907,
 10905,
 10904,
 10950,
 10945,
 10951,
 10906,
 11186,
 10952,
 10954,
 10953,
 10779,
 10957,
 10956,
 10955,
 10959,
 11198,
 10958,
 11026,
 11110,
 11138,
 11136,
 11134,
 11124,
 11135,
 11123,
 11137,
 11133,
 11130,
 11139,
 11131,
 11129,
 11132,
 11128,
 11122,
 11127,
 11121,
 11125,
 11120,
 11140,
 11119,
 11126,
 11142,
 11141,
 11143,
 11145,
 11149,
 11118,
 11144,
 11113,
 11146,
 111

✅ Done displaying.

In [4]:
# =========================
# Counts of intersection matches per latent (by layer)
# (does NOT rely on code_size_by_layer)
# =========================
from collections import Counter
import pandas as pd
from IPython.display import display, Markdown
import numpy as np
import torch

def _infer_total_latents_for_layer(L: int) -> int:
    """
    Tries to infer total latent count (d_sae) from objects already in memory.
    Fallback: uses max latent_id seen in rankings + 1 (works if rankings are full over all latents).
    """
    # 1) If you cached per-layer diffs (best)
    if "diff_by_layer_dataset" in globals():
        d = diff_by_layer_dataset.get(L, None)
        if d is not None:
            if isinstance(d, torch.Tensor):
                return int(d.numel())
            if isinstance(d, np.ndarray):
                return int(d.size)

    # 2) If you have any full ranking list (common)
    if "ranked_by_layer_dataset" in globals():
        any_list = next(iter(ranked_by_layer_dataset[L].values()))
        return int(len(any_list))

    # 3) Fallback: max latent id observed + 1
    mx = -1
    if "ranked_by_layer_dataset" in globals():
        for lst in ranked_by_layer_dataset[L].values():
            mx = max(mx, int(max(lst)))
    if mx >= 0:
        return mx + 1

    raise NameError(
        "Couldn't infer total latent count. "
        "Make sure you have either diff_by_layer_dataset[layer] or ranked_by_layer_dataset[layer] in memory."
    )

def _counts_table(items, total_latents: int):
    # items: list[(latent_id, count, rank_sum)] for latents with count>=1 (in top-N of at least one list)
    c = Counter()
    for _, cnt, _ in items:
        c[int(cnt)] += 1

    nonzero = sum(c.values())
    c[0] = total_latents - nonzero

    rows = [{"matches": k, "n_latents": c[k]} for k in sorted(c.keys())]
    df = pd.DataFrame(rows)
    df["frac"] = df["n_latents"] / float(total_latents)
    return df

for L in SAE_LAYERS:
    total_latents = _infer_total_latents_for_layer(L)

    df_ds = _counts_table(intersection_by_layer_dataset[L]["items"], total_latents)
    df_sp = _counts_table(intersection_by_layer_split[L]["items"], total_latents)

    display(Markdown(f"### Layer {L} — match-count distribution (top-{INTERSECTION_TOP_N} membership)"))
    display(Markdown(f"**Total latents:** {total_latents}"))

    display(Markdown("**Across datasets (4 lists):**"))
    display(df_ds)

    display(Markdown("**Across splits (13 lists):**"))
    display(df_sp)


### Layer 9 — match-count distribution (top-1000 membership)

**Total latents:** 16384

**Across datasets (4 lists):**

,matches,n_latents,frac
0,0,15027,0.917175
1,1,263,0.016052
2,2,178,0.010864
3,3,283,0.017273
4,4,633,0.038635


**Across splits (13 lists):**

...

**Total latents:** 16384

**Across datasets (4 lists):**

,matches,n_latents,frac
0,0,14843,0.905945
1,1,460,0.028076
2,2,143,0.008728
3,3,498,0.030396
4,4,440,0.026855


**Across splits (13 lists):**

,matches,n_latents,frac
0,0,14805,0.903625
1,1,58,0.003540
2,2,311,0.018982
3,3,103,0.006287
4,4,63,0.003845
5,5,42,0.002563
6,6,42,0.002563
7,7,26,0.001587
8,8,47,0.002869
9,9,45,0.002747


In [3]:
# Pairwise shared-latent matrix + per-split unique counts
# Assumes ranked_by_layer_split is already in memory from the previous code.

import pandas as pd
from IPython.display import display, Markdown

# --- Config: which splits to include (11) ---
SPLITS_FOR_MATRIX = [
    "coconot_cat_humanizing_requests",
    "coconot_cat_incomplete_requests",
    "coconot_cat_indeterminate_requests",
    "coconot_cat_requests_with_safety_concerns",
    "coconot_cat_unsupported_requests",
    "sorrybench_hate_speech_generation",
    "sorrybench_assistance_with_crimes_or_torts",
    "sorrybench_potentially_inappropriate_topics",
    "sorrybench_potentially_unqualified_advice",
    "wildguard_all",
    "xstest_all",
]

# Infer layers from ranked_by_layer_split if SAE_LAYERS is not defined.
try:
    layers = list(SAE_LAYERS)
except NameError:
    layers = sorted(ranked_by_layer_split.keys())

# Use the same top-N you used for intersections, or default to 1000.
try:
    TOPN_FOR_MATRIX = int(INTERSECTION_TOP_N)
except NameError:
    TOPN_FOR_MATRIX = 1000

# Sanity check: make sure all splits exist for at least one layer
example_layer = layers[0]
missing = [s for s in SPLITS_FOR_MATRIX if s not in ranked_by_layer_split[example_layer]]
if missing:
    raise ValueError(f"The following splits are missing from ranked_by_layer_split[{example_layer}]: {missing}")

for L in layers:
    display(Markdown(f"## Layer {L} — shared-top{TOPN_FOR_MATRIX} latent counts"))

    # --- Build top-N sets per split ---
    topn_sets = {}
    for split_name in SPLITS_FOR_MATRIX:
        ranking = ranked_by_layer_split[L][split_name]
        # take the first TOPN_FOR_MATRIX latent IDs (or fewer if ranking is shorter)
        topn_sets[split_name] = set(ranking[:TOPN_FOR_MATRIX])

    # --- Build 11×11 matrix of shared counts ---
    matrix_data = []
    for row_split in SPLITS_FOR_MATRIX:
        row_vals = []
        for col_split in SPLITS_FOR_MATRIX:
            shared = topn_sets[row_split] & topn_sets[col_split]
            row_vals.append(len(shared))
        matrix_data.append(row_vals)

    df_shared = pd.DataFrame(
        matrix_data,
        index=SPLITS_FOR_MATRIX,
        columns=SPLITS_FOR_MATRIX,
    )

    display(Markdown(f"**Pairwise counts of shared latents in the top {TOPN_FOR_MATRIX} for each split pair**"))
    display(df_shared)

    # --- Unique latents per split (within top-N) ---
    unique_counts = {}
    for split_name in SPLITS_FOR_MATRIX:
        others_union = set()
        for other in SPLITS_FOR_MATRIX:
            if other == split_name:
                continue
            others_union |= topn_sets[other]
        unique_latents = topn_sets[split_name] - others_union
        unique_counts[split_name] = len(unique_latents)

    df_unique = pd.DataFrame.from_dict(
        unique_counts,
        orient="index",
        columns=[f"unique_latents_in_top{TOPN_FOR_MATRIX}"],
    )

    display(Markdown("**Number of latents that only appear in this split's top-N (and in no other split's top-N):**"))
    display(df_unique.sort_index())


## Layer 9 — shared-top1000 latent counts

**Pairwise counts of shared latents in the top 1000 for each split pair**

,coconot_cat_humanizing_requests,coconot_cat_incomplete_requests,coconot_cat_indeterminate_requests,coconot_cat_requests_with_safety_concerns,coconot_cat_unsupported_requests,sorrybench_hate_speech_generation,sorrybench_assistance_with_crimes_or_torts,sorrybench_potentially_inappropriate_topics,sorrybench_potentially_unqualified_advice,wildguard_all,xstest_all
coconot_cat_humanizing_requests,1000,915,937,797,845,963,912,948,973,961,956
coconot_cat_incomplete_requests,915,1000,923,819,876,911,915,919,914,916,904
coconot_cat_indeterminate_requests,937,923,1000,799,859,930,933,934,932,934,918
coconot_cat_requests_with_safety_concerns,797,819,799,1000,829,797,806,813,796,801,787
coconot_cat_unsupported_requests,845,876,859,829,1000,841,863,854,843,845,838
sorrybench_hate_speech_generation,963,911,930,797,841,1000,913,961,962,964,958
sorrybench_assistance_with_crimes_or_torts,912,915,933,806,863,913,1000,928,913,914,896
sorrybench_potentially_inappropriate_topics,948,919,934,813,854,961,928,1000,949,959,936
sorrybench_potentially_unqualified_advice,973,914,932,796,843,962,913,949,1000,964,955
wildguard_all,961,916,934,801,845,964,914,959,964,1000,951


**Number of latents that only appear in this split's top-N (and in no other split's top-N):**

,unique_latents_in_top1000
coconot_cat_humanizing_requests,5
coconot_cat_incomplete_requests,16
coconot_cat_indeterminate_requests,10
coconot_cat_requests_with_safety_concerns,79
coconot_cat_unsupported_requests,51
sorrybench_assistance_with_crimes_or_torts,19
sorrybench_hate_speech_generation,2
sorrybench_potentially_inappropriate_topics,4
sorrybench_potentially_unqualified_advice,4
wildguard_all,5


...

## Layer 31 — shared-top1000 latent counts

**Pairwise counts of shared latents in the top 1000 for each split pair**

,coconot_cat_humanizing_requests,coconot_cat_incomplete_requests,coconot_cat_indeterminate_requests,coconot_cat_requests_with_safety_concerns,coconot_cat_unsupported_requests,sorrybench_hate_speech_generation,sorrybench_assistance_with_crimes_or_torts,sorrybench_potentially_inappropriate_topics,sorrybench_potentially_unqualified_advice,wildguard_all,xstest_all
coconot_cat_humanizing_requests,1000,816,942,760,838,937,933,918,944,918,931
coconot_cat_incomplete_requests,816,1000,820,783,833,805,804,805,803,816,798
coconot_cat_indeterminate_requests,942,820,1000,768,845,935,935,927,925,925,920
coconot_cat_requests_with_safety_concerns,760,783,768,1000,776,766,773,785,747,797,731
coconot_cat_unsupported_requests,838,833,845,776,1000,820,821,829,823,842,822
sorrybench_hate_speech_generation,937,805,935,766,820,1000,964,941,948,935,943
sorrybench_assistance_with_crimes_or_torts,933,804,935,773,821,964,1000,940,945,944,933
sorrybench_potentially_inappropriate_topics,918,805,927,785,829,941,940,1000,915,950,903
sorrybench_potentially_unqualified_advice,944,803,925,747,823,948,945,915,1000,917,964
wildguard_all,918,816,925,797,842,935,944,950,917,1000,902


**Number of latents that only appear in this split's top-N (and in no other split's top-N):**

,unique_latents_in_top1000
coconot_cat_humanizing_requests,12
coconot_cat_incomplete_requests,106
coconot_cat_indeterminate_requests,15
coconot_cat_requests_with_safety_concerns,91
coconot_cat_unsupported_requests,73
sorrybench_assistance_with_crimes_or_torts,8
sorrybench_hate_speech_generation,6
sorrybench_potentially_inappropriate_topics,17
sorrybench_potentially_unqualified_advice,6
wildguard_all,12


In [5]:
# Shared-topN matrix and unique counts **across all layers**
# Assumes `ranked_by_layer_split` is already populated from your earlier code.

import pandas as pd
from IPython.display import display, Markdown

# --- Config: which splits to include (11) ---
SPLITS_FOR_MATRIX = [
    "coconot_cat_humanizing_requests",
    "coconot_cat_incomplete_requests",
    "coconot_cat_indeterminate_requests",
    "coconot_cat_requests_with_safety_concerns",
    "coconot_cat_unsupported_requests",
    "sorrybench_hate_speech_generation",
    "sorrybench_assistance_with_crimes_or_torts",
    "sorrybench_potentially_inappropriate_topics",
    "sorrybench_potentially_unqualified_advice",
    "wildguard_all",
    "xstest_all",
]

# Infer layers from SAE_LAYERS if available, otherwise from ranked_by_layer_split
try:
    layers = list(SAE_LAYERS)
except NameError:
    layers = sorted(ranked_by_layer_split.keys())

# Use the same top-N you used for intersections, or default to 1000.
try:
    TOPN_FOR_MATRIX = int(INTERSECTION_TOP_N)
except NameError:
    TOPN_FOR_MATRIX = 1000

# Sanity: make sure all splits exist for every layer we plan to use
for L in layers:
    missing = [s for s in SPLITS_FOR_MATRIX if s not in ranked_by_layer_split[L]]
    if missing:
        raise ValueError(f"Layer {L} is missing splits: {missing}")

display(Markdown(
    f"# Shared-latent statistics across all layers\n"
    f"- Top-N per split *per layer*: **{TOPN_FOR_MATRIX}**\n"
    f"- Layers included: {layers}"
))

# --- Build per-split union sets across layers ---
# Each element is a tuple (layer, latent_id) so latents from different layers don't collide.
union_sets = {split: set() for split in SPLITS_FOR_MATRIX}

for L in layers:
    for split_name in SPLITS_FOR_MATRIX:
        ranking = ranked_by_layer_split[L][split_name]
        top_ids = ranking[:TOPN_FOR_MATRIX]
        union_sets[split_name].update((L, int(latent_id)) for latent_id in top_ids)

# Optional: show total number of (layer, latent) pairs considered per split
totals = {s: len(union_sets[s]) for s in SPLITS_FOR_MATRIX}
df_totals = pd.DataFrame.from_dict(
    totals, orient="index", columns=[f"n_(layer,latent)_pairs_in_union"]
).sort_index()

display(Markdown("## Total (layer, latent) pairs per split (union across layers)"))
display(df_totals)

# --- Build 11×11 matrix of shared counts across all layers ---
matrix_data = []
for row_split in SPLITS_FOR_MATRIX:
    row_vals = []
    for col_split in SPLITS_FOR_MATRIX:
        shared = union_sets[row_split] & union_sets[col_split]
        row_vals.append(len(shared))
    matrix_data.append(row_vals)

df_shared_all_layers = pd.DataFrame(
    matrix_data,
    index=SPLITS_FOR_MATRIX,
    columns=SPLITS_FOR_MATRIX,
)

display(Markdown(
    f"## Pairwise shared-latent counts across all layers\n"
    f"Each entry (i, j) = number of (layer, latent_id) pairs that are in the top {TOPN_FOR_MATRIX} "
    f"for **both** splits i and j (possibly from any layer)."
))
display(df_shared_all_layers)

# --- Unique latents per split (within union across layers) ---
unique_counts = {}
for split_name in SPLITS_FOR_MATRIX:
    others_union = set()
    for other in SPLITS_FOR_MATRIX:
        if other == split_name:
            continue
        others_union |= union_sets[other]
    unique_latents = union_sets[split_name] - others_union
    unique_counts[split_name] = len(unique_latents)

df_unique_all_layers = pd.DataFrame.from_dict(
    unique_counts,
    orient="index",
    columns=["unique_(layer,latent)_pairs"],
).sort_index()

display(Markdown(
    "## Unique latents per split across all layers\n"
    "- A pair (layer, latent_id) is counted as unique for a split if it appears in that split's "
    "top-N for **some layer**, and does **not** appear in the top-N of any other split for any layer."
))
display(df_unique_all_layers)


# Shared-latent statistics across all layers
- Top-N per split *per layer*: **1000**
- Layers included: [9, 20, 31]

## Total (layer, latent) pairs per split (union across layers)

,"n_(layer,latent)_pairs_in_union"
coconot_cat_humanizing_requests,3000
coconot_cat_incomplete_requests,3000
coconot_cat_indeterminate_requests,3000
coconot_cat_requests_with_safety_concerns,3000
coconot_cat_unsupported_requests,3000
sorrybench_assistance_with_crimes_or_torts,3000
sorrybench_hate_speech_generation,3000
sorrybench_potentially_inappropriate_topics,3000
sorrybench_potentially_unqualified_advice,3000
wildguard_all,3000


## Pairwise shared-latent counts across all layers
Each entry (i, j) = number of (layer, latent_id) pairs that are in the top 1000 for **both** splits i and j (possibly from any layer).

,coconot_cat_humanizing_requests,coconot_cat_incomplete_requests,coconot_cat_indeterminate_requests,coconot_cat_requests_with_safety_concerns,coconot_cat_unsupported_requests,sorrybench_hate_speech_generation,sorrybench_assistance_with_crimes_or_torts,sorrybench_potentially_inappropriate_topics,sorrybench_potentially_unqualified_advice,wildguard_all,xstest_all
coconot_cat_humanizing_requests,3000,2594,2836,2346,2517,2836,2789,2811,2869,2811,2823
coconot_cat_incomplete_requests,2594,3000,2605,2395,2551,2565,2567,2571,2569,2588,2551
coconot_cat_indeterminate_requests,2836,2605,3000,2347,2546,2816,2816,2813,2804,2794,2789
coconot_cat_requests_with_safety_concerns,2346,2395,2347,3000,2398,2334,2368,2386,2323,2401,2285
coconot_cat_unsupported_requests,2517,2551,2546,2398,3000,2485,2504,2509,2492,2522,2480
sorrybench_hate_speech_generation,2836,2565,2816,2334,2485,3000,2828,2862,2866,2835,2878
sorrybench_assistance_with_crimes_or_torts,2789,2567,2816,2368,2504,2828,3000,2831,2815,2809,2784
sorrybench_potentially_inappropriate_topics,2811,2571,2813,2386,2509,2862,2831,3000,2824,2863,2793
sorrybench_potentially_unqualified_advice,2869,2569,2804,2323,2492,2866,2815,2824,3000,2825,2877
wildguard_all,2811,2588,2794,2401,2522,2835,2809,2863,2825,3000,2787


## Unique latents per split across all layers
- A pair (layer, latent_id) is counted as unique for a split if it appears in that split's top-N for **some layer**, and does **not** appear in the top-N of any other split for any layer.

,"unique_(layer,latent)_pairs"
coconot_cat_humanizing_requests,27
coconot_cat_incomplete_requests,165
coconot_cat_indeterminate_requests,32
coconot_cat_requests_with_safety_concerns,257
coconot_cat_unsupported_requests,177
sorrybench_assistance_with_crimes_or_torts,34
sorrybench_hate_speech_generation,11
sorrybench_potentially_inappropriate_topics,27
sorrybench_potentially_unqualified_advice,17
wildguard_all,24


In [1]:
# =========================
# Count active SAE latents at layer 31
# (over the same filtered data as the main experiment)
# =========================

import json
from pathlib import Path
import gc

import numpy as np
import torch
import torch.nn as nn

# -----------------------
# CONFIG
# -----------------------
DTYPE = torch.float16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAE config (edit paths to match your setup)
SAE_BASE_DIR = Path("../exp1_v3/saes")  # <- change if needed
SAE_WIDTH = "16k"
SAE_L0 = "14"
SAE_LAYERS = [9, 20, 31]          # Gemma-Scope convention
BLOCK_IDXS = [l + 1 for l in SAE_LAYERS]  # [10, 21, 32]
POS = -2                          # not used here but kept for consistency

# Layer we care about
TARGET_LAYER = 31

# Generation JSONs (same as original)
GEN_COCONOT = Path("generations/coconot/original/evaluated/train_generations_wildguard.json")
GEN_SORRY   = Path("generations/sorrybench/evaluated/sorrybench_base440_generations_wildguard.json")
GEN_WG      = Path("generations/original_wildguard/evaluated/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_wildguard.json")
GEN_XS      = Path("generations/xstest/evaluated/test_generations_wildguard.json")

# Activation caches (same as original)
ACTS_COCONOT = Path("activations/coconot/original/train_l10_21_32_pos-2_residpre.pt")
ACTS_SORRY   = Path("activations/sorrybench_base440_l10_21_32_pos-2_residpre.pt")
ACTS_WG      = Path("activations/recomputed/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_l10_21_32_pos-2_residpre.pt")
ACTS_XS      = Path("activations/xstest/test_l10_21_32_pos-2_residpre.pt")


# -----------------------
# SAE DEFINITION + LOADING
# -----------------------
class JumpReLUSAE(nn.Module):
    def __init__(self, d_model, d_sae):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.threshold = nn.Parameter(torch.zeros(d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        mask = pre > self.threshold
        return mask * torch.relu(pre)

def load_local_sae(layer: int, device=DEVICE, dtype=DTYPE):
    params_path = (
        SAE_BASE_DIR
        / f"layer_{layer}"
        / f"width_{SAE_WIDTH}"
        / f"average_l0_{SAE_L0}"
        / "params.npz"
    )
    params = np.load(params_path)
    sae = JumpReLUSAE(params["W_enc"].shape[0], params["W_enc"].shape[1]).to(device, dtype)
    sae.W_enc.data.copy_(torch.from_numpy(params["W_enc"]).to(device, dtype))
    sae.W_dec.data.copy_(torch.from_numpy(params["W_dec"]).to(device, dtype))
    sae.b_enc.data.copy_(torch.from_numpy(params["b_enc"]).to(device, dtype))
    sae.b_dec.data.copy_(torch.from_numpy(params["b_dec"]).to(device, dtype))
    sae.threshold.data.copy_(torch.from_numpy(params["threshold"]).to(device, dtype))
    sae.eval()
    return sae


# -----------------------
# HELPERS: LOAD + ALIGN DATA (same filtering as original)
# -----------------------
def parse_binary_is_refusal(val):
    """Return 0/1 or None (drops 'n/a', None, other junk)."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, (int, np.integer)):
        return int(val) if val in (0, 1) else None
    if isinstance(val, float):
        return int(val) if val in (0.0, 1.0) else None
    if isinstance(val, str):
        s = val.strip().lower()
        if s in ("0", "1"):
            return int(s)
        return None
    return None

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def load_acts(path: Path):
    obj = torch.load(path, map_location="cpu")
    # minimal support: either dict-with-acts or raw tensor
    if isinstance(obj, dict) and "acts" in obj:
        acts = obj["acts"]
        meta = {k: v for k, v in obj.items() if k != "acts"}
        return acts, meta
    if torch.is_tensor(obj):
        return obj, {}
    raise ValueError(f"Activation file not understood: {path}")

def align_records_and_acts(records, acts):
    """
    Drops non-binary is_refusal rows from records,
    and drops the same rows from acts (assumes acts aligned to original record order).
    """
    keep = []
    y = []
    for i, r in enumerate(records):
        v = parse_binary_is_refusal(r.get("is_refusal", None))
        if v is None:
            continue
        keep.append(i)
        y.append(v)

    keep = torch.tensor(keep, dtype=torch.long)

    # acts can be [N, D] or [N, L, D]
    if acts.size(0) == len(records):
        acts_kept = acts.index_select(0, keep)
    elif acts.size(0) == keep.numel():
        acts_kept = acts  # already filtered upstream
    else:
        raise ValueError(f"Length mismatch: records={len(records)} but acts[0]={acts.size(0)}")

    recs_kept = [records[i] for i in keep.tolist()]
    y = torch.tensor(y, dtype=torch.long)
    return recs_kept, y, acts_kept

def select_layer_slice(acts, layer: int, meta: dict):
    """
    Returns [N, D] for a given SAE layer from multi-layer cache [N, L, D].
    Assumes ordering matches BLOCK_IDXS = [10,21,32] (= SAE_LAYERS+1).
    """
    if acts.dim() == 2:
        return acts
    assert acts.dim() == 3, f"Expected acts [N,L,D] or [N,D], got {tuple(acts.shape)}"

    # if metadata provides block_idxs, use it; otherwise assume [10,21,32]
    block_idxs = meta.get("block_idxs", BLOCK_IDXS)
    if isinstance(block_idxs, torch.Tensor):
        block_idxs = block_idxs.tolist()

    wanted_block = layer + 1
    if wanted_block not in block_idxs:
        raise ValueError(f"wanted block {wanted_block} not in block_idxs={block_idxs}")
    col = block_idxs.index(wanted_block)
    return acts[:, col, :]


# -----------------------
# MAIN: COUNT EVER-ACTIVE LATENTS AT TARGET_LAYER
# -----------------------
def main():
    # 1) Load records + activations
    cn_records_raw = load_json(GEN_COCONOT)
    sb_records_raw = load_json(GEN_SORRY)
    wg_records_raw = load_json(GEN_WG)
    xs_records_raw = load_json(GEN_XS)

    cn_acts_raw, cn_meta = load_acts(ACTS_COCONOT)
    sb_acts_raw, sb_meta = load_acts(ACTS_SORRY)
    wg_acts_raw, wg_meta = load_acts(ACTS_WG)
    xs_acts_raw, xs_meta = load_acts(ACTS_XS)

    cn_records, cn_isref, cn_acts = align_records_and_acts(cn_records_raw, cn_acts_raw)
    sb_records, sb_isref, sb_acts = align_records_and_acts(sb_records_raw, sb_acts_raw)
    wg_records, wg_isref, wg_acts = align_records_and_acts(wg_records_raw, wg_acts_raw)
    xs_records, xs_isref, xs_acts = align_records_and_acts(xs_records_raw, xs_acts_raw)

    print("After dropping non-binary is_refusal rows:")
    print(f"  coconot:    {len(cn_records)}")
    print(f"  sorrybench: {len(sb_records)}")
    print(f"  wildguard:  {len(wg_records)}")
    print(f"  xstest:     {len(xs_records)}")

    # Pack acts/meta for convenience
    ACTS_BY_SRC = {
        "coconot":   (cn_acts, cn_meta),
        "sorrybench":(sb_acts, sb_meta),
        "wildguard": (wg_acts, wg_meta),
        "xstest":    (xs_acts, xs_meta),
    }

    # 2) Load SAE for target layer
    L = TARGET_LAYER
    print(f"\nCounting active latents for layer {L}...")
    sae = load_local_sae(L)

    # 3) Collect all activations for layer L across all 4 datasets
    acts2d_all = []
    for src, (acts, meta) in ACTS_BY_SRC.items():
        acts2d = select_layer_slice(acts, L, meta)  # [N_src, D] CPU
        acts2d_all.append(acts2d)

    acts2d_all = torch.cat(acts2d_all, dim=0)  # [N_total, D_model]
    print(f"Total examples across all datasets: {acts2d_all.size(0)}")

    # 4) Run SAE encode in batches and track "ever active > 0" per latent
    code_size = sae.W_enc.shape[1]  # number of SAE latents
    ever_active = torch.zeros(code_size, dtype=torch.bool)  # [code_size]

    BATCH = 512  # adjust as needed

    for start in range(0, acts2d_all.size(0), BATCH):
        end = start + BATCH
        x = acts2d_all[start:end].to(DEVICE, dtype=sae.W_enc.dtype)

        with torch.no_grad():
            z = sae.encode(x)           # [batch, code_size]
            active_batch = (z > 0)      # bool mask

        ever_active |= active_batch.any(dim=0).cpu()

        del x, z, active_batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 5) Count latents
    num_ever_active = int(ever_active.sum().item())
    total_latents   = int(ever_active.numel())
    num_always_zero = total_latents - num_ever_active

    print(f"\nLayer {L} SAE latents:")
    print(f"  Total latents:           {total_latents}")
    print(f"  Latents ever active > 0: {num_ever_active}")
    print(f"  Latents always 0:        {num_always_zero}")

    # Optional: list of active latent IDs
    # active_ids = ever_active.nonzero(as_tuple=True)[0].tolist()
    # print(f"Example active ids (first 50): {active_ids[:50]}")

    # cleanup
    del sae, acts2d_all
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


After dropping non-binary is_refusal rows:
  coconot:    11473
  sorrybench: 440
  wildguard:  915
  xstest:     450

Counting active latents for layer 31...
Total examples across all datasets: 13278

Layer 31 SAE latents:
  Total latents:           16384
  Latents ever active > 0: 1086
  Latents always 0:        15298


In [1]:
# =========================
# Count active SAE latents at layer 31
# Using ONLY HR + BC examples (same definitions as main experiment)
# =========================

import json
from pathlib import Path
import gc

import numpy as np
import torch
import torch.nn as nn

# -----------------------
# CONFIG
# -----------------------
DTYPE = torch.float16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAE config (match your main script)
SAE_BASE_DIR = Path("../exp1_v3/saes")  # <- change if needed
SAE_WIDTH = "16k"
SAE_L0 = "14"
SAE_LAYERS = [9, 20, 31]          # Gemma-Scope convention
BLOCK_IDXS = [l + 1 for l in SAE_LAYERS]  # [10, 21, 32]

# Layer we care about
TARGET_LAYER = 31

# Generation JSONs (same as original)
GEN_COCONOT = Path("generations/coconot/original/evaluated/train_generations_wildguard.json")
GEN_SORRY   = Path("generations/sorrybench/evaluated/sorrybench_base440_generations_wildguard.json")
GEN_WG      = Path("generations/original_wildguard/evaluated/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_wildguard.json")
GEN_XS      = Path("generations/xstest/evaluated/test_generations_wildguard.json")

# Activation caches (same as original)
ACTS_COCONOT = Path("activations/coconot/original/train_l10_21_32_pos-2_residpre.pt")
ACTS_SORRY   = Path("activations/sorrybench_base440_l10_21_32_pos-2_residpre.pt")
ACTS_WG      = Path("activations/recomputed/wildguardtest_nonadv_wildguardtest_nonadversarial_generations_l10_21_32_pos-2_residpre.pt")
ACTS_XS      = Path("activations/xstest/test_l10_21_32_pos-2_residpre.pt")


# -----------------------
# SAE DEFINITION + LOADING
# -----------------------
class JumpReLUSAE(nn.Module):
    def __init__(self, d_model, d_sae):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.threshold = nn.Parameter(torch.zeros(d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        pre = x @ self.W_enc + self.b_enc
        mask = pre > self.threshold
        return mask * torch.relu(pre)


def load_local_sae(layer: int, device=DEVICE, dtype=DTYPE):
    params_path = (
        SAE_BASE_DIR
        / f"layer_{layer}"
        / f"width_{SAE_WIDTH}"
        / f"average_l0_{SAE_L0}"
        / "params.npz"
    )
    params = np.load(params_path)
    sae = JumpReLUSAE(params["W_enc"].shape[0], params["W_enc"].shape[1]).to(device, dtype)
    sae.W_enc.data.copy_(torch.from_numpy(params["W_enc"]).to(device, dtype))
    sae.W_dec.data.copy_(torch.from_numpy(params["W_dec"]).to(device, dtype))
    sae.b_enc.data.copy_(torch.from_numpy(params["b_enc"]).to(device, dtype))
    sae.b_dec.data.copy_(torch.from_numpy(params["b_dec"]).to(device, dtype))
    sae.threshold.data.copy_(torch.from_numpy(params["threshold"]).to(device, dtype))
    sae.eval()
    return sae


# -----------------------
# HELPERS: LOAD + ALIGN DATA
# -----------------------
def parse_binary_is_refusal(val):
    """Return 0/1 or None (drops 'n/a', None, other junk)."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, (int, np.integer)):
        return int(val) if val in (0, 1) else None
    if isinstance(val, float):
        return int(val) if val in (0.0, 1.0) else None
    if isinstance(val, str):
        s = val.strip().lower()
        if s in ("0", "1"):
            return int(s)
        return None
    return None


def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)


def load_acts(path: Path):
    obj = torch.load(path, map_location="cpu")
    # minimal support: either dict-with-acts or raw tensor
    if isinstance(obj, dict) and "acts" in obj:
        acts = obj["acts"]
        meta = {k: v for k, v in obj.items() if k != "acts"}
        return acts, meta
    if torch.is_tensor(obj):
        return obj, {}
    raise ValueError(f"Activation file not understood: {path}")


def align_records_and_acts(records, acts):
    """
    Drops non-binary is_refusal rows from records,
    and drops the same rows from acts (assumes acts aligned to original record order).
    """
    keep = []
    y = []
    for i, r in enumerate(records):
        v = parse_binary_is_refusal(r.get("is_refusal", None))
        if v is None:
            continue
        keep.append(i)
        y.append(v)

    keep = torch.tensor(keep, dtype=torch.long)

    # acts can be [N, D] or [N, L, D]
    if acts.size(0) == len(records):
        acts_kept = acts.index_select(0, keep)
    elif acts.size(0) == keep.numel():
        acts_kept = acts  # already filtered upstream
    else:
        raise ValueError(f"Length mismatch: records={len(records)} but acts[0]={acts.size(0)}")

    recs_kept = [records[i] for i in keep.tolist()]
    y = torch.tensor(y, dtype=torch.long)
    return recs_kept, y, acts_kept


def select_layer_slice(acts, layer: int, meta: dict):
    """
    Returns [N, D] for a given SAE layer from multi-layer cache [N, L, D].
    Assumes ordering matches BLOCK_IDXS = [10,21,32].
    """
    if acts.dim() == 2:
        return acts
    assert acts.dim() == 3, f"Expected acts [N,L,D] or [N,D], got {tuple(acts.shape)}"

    # if metadata provides block_idxs, use it; otherwise assume [10,21,32]
    block_idxs = meta.get("block_idxs", BLOCK_IDXS)
    if isinstance(block_idxs, torch.Tensor):
        block_idxs = block_idxs.tolist()

    wanted_block = layer + 1
    if wanted_block not in block_idxs:
        raise ValueError(f"wanted block {wanted_block} not in block_idxs={block_idxs}")
    col = block_idxs.index(wanted_block)
    return acts[:, col, :]


# -----------------------
# HR / BC INDEX HELPERS (same definitions as main experiment)
# -----------------------
def idx_wg_benign_compliant(records, isref):
    # benign/unharmful interchangeable: prompt_harm_label == "unharmful"
    m = []
    for i, r in enumerate(records):
        if (r.get("prompt_harm_label") == "unharmful") and (int(isref[i]) == 0):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)


def idx_wg_harmful_refusal(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("prompt_harm_label") == "harmful") and (int(isref[i]) == 1):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)


def idx_xs_benign_compliant(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("label") == "safe") and (int(isref[i]) == 0):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)


def idx_xs_harmful_refusal(records, isref):
    m = []
    for i, r in enumerate(records):
        if (r.get("label") == "unsafe") and (int(isref[i]) == 1):
            m.append(i)
    return torch.tensor(m, dtype=torch.long)


def idx_all_refusals(isref):
    return (isref == 1).nonzero(as_tuple=True)[0]


# -----------------------
# MAIN
# -----------------------
def main():
    # 1) Load records + activations
    cn_records_raw = load_json(GEN_COCONOT)
    sb_records_raw = load_json(GEN_SORRY)
    wg_records_raw = load_json(GEN_WG)
    xs_records_raw = load_json(GEN_XS)

    cn_acts_raw, cn_meta = load_acts(ACTS_COCONOT)
    sb_acts_raw, sb_meta = load_acts(ACTS_SORRY)
    wg_acts_raw, wg_meta = load_acts(ACTS_WG)
    xs_acts_raw, xs_meta = load_acts(ACTS_XS)

    cn_records, cn_isref, cn_acts = align_records_and_acts(cn_records_raw, cn_acts_raw)
    sb_records, sb_isref, sb_acts = align_records_and_acts(sb_records_raw, sb_acts_raw)
    wg_records, wg_isref, wg_acts = align_records_and_acts(wg_records_raw, wg_acts_raw)
    xs_records, xs_isref, xs_acts = align_records_and_acts(xs_records_raw, xs_acts_raw)

    print("After dropping non-binary is_refusal rows:")
    print(f"  coconot:    {len(cn_records)}")
    print(f"  sorrybench: {len(sb_records)}")
    print(f"  wildguard:  {len(wg_records)}")
    print(f"  xstest:     {len(xs_records)}")

    # 2) Build HR / BC indices
    # HR (harmful refusals)
    cn_hr_idx = idx_all_refusals(cn_isref)          # all refusals in coconot (harmful-only)
    sb_hr_idx = idx_all_refusals(sb_isref)          # all refusals in sorrybench (harmful-only)
    wg_hr_idx = idx_wg_harmful_refusal(wg_records, wg_isref)
    xs_hr_idx = idx_xs_harmful_refusal(xs_records, xs_isref)

    # BC (benign compliance)
    wg_bc_idx = idx_wg_benign_compliant(wg_records, wg_isref)
    xs_bc_idx = idx_xs_benign_compliant(xs_records, xs_isref)

    # 3) Load SAE for target layer
    L = TARGET_LAYER
    print(f"\nCounting active latents for layer {L} (HR + BC only)...")
    sae = load_local_sae(L)

    # 4) Slice activations at layer L and restrict to HR+BC rows, across all datasets
    acts2d_all = []

    # Helper to safely append only if indices non-empty
    def _append_if_nonempty(tensor_2d, idx):
        if idx.numel() == 0:
            return
        acts2d_all.append(tensor_2d.index_select(0, idx))

    # coconot: HR only
    acts_cn_L = select_layer_slice(cn_acts, L, cn_meta)
    _append_if_nonempty(acts_cn_L, cn_hr_idx)

    # sorrybench: HR only
    acts_sb_L = select_layer_slice(sb_acts, L, sb_meta)
    _append_if_nonempty(acts_sb_L, sb_hr_idx)

    # wildguard: HR + BC
    acts_wg_L = select_layer_slice(wg_acts, L, wg_meta)
    if wg_hr_idx.numel() > 0 and wg_bc_idx.numel() > 0:
        wg_keep_idx = torch.cat([wg_hr_idx, wg_bc_idx], dim=0)
    elif wg_hr_idx.numel() > 0:
        wg_keep_idx = wg_hr_idx
    else:
        wg_keep_idx = wg_bc_idx
    _append_if_nonempty(acts_wg_L, wg_keep_idx)

    # xstest: HR + BC
    acts_xs_L = select_layer_slice(xs_acts, L, xs_meta)
    if xs_hr_idx.numel() > 0 and xs_bc_idx.numel() > 0:
        xs_keep_idx = torch.cat([xs_hr_idx, xs_bc_idx], dim=0)
    elif xs_hr_idx.numel() > 0:
        xs_keep_idx = xs_hr_idx
    else:
        xs_keep_idx = xs_bc_idx
    _append_if_nonempty(acts_xs_L, xs_keep_idx)

    if len(acts2d_all) == 0:
        print("No HR/BC examples found. Nothing to encode.")
        return

    acts2d_all = torch.cat(acts2d_all, dim=0)  # [N_total_HRBC, D_model]
    print(f"Total HR+BC examples across all datasets: {acts2d_all.size(0)}")

    # 5) Run SAE encode in batches and track which latents were ever active (>0)
    code_size = sae.W_enc.shape[1]  # number of SAE latents
    ever_active = torch.zeros(code_size, dtype=torch.bool)  # [code_size]

    BATCH = 512  # adjust as needed

    for start in range(0, acts2d_all.size(0), BATCH):
        end = start + BATCH
        x = acts2d_all[start:end].to(DEVICE, dtype=sae.W_enc.dtype)

        with torch.no_grad():
            z = sae.encode(x)           # [batch, code_size]
            active_batch = (z > 0)      # bool mask

        ever_active |= active_batch.any(dim=0).cpu()

        del x, z, active_batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 6) Count latents
    num_ever_active = int(ever_active.sum().item())
    total_latents   = int(ever_active.numel())
    num_always_zero = total_latents - num_ever_active

    print(f"\nLayer {L} SAE latents (HR+BC examples only):")
    print(f"  Total latents:              {total_latents}")
    print(f"  Latents ever active  (> 0): {num_ever_active}")
    print(f"  Latents always zero (== 0): {num_always_zero}")

    # Optional: list of active latent IDs
    # active_ids = ever_active.nonzero(as_tuple=True)[0].tolist()
    # print(f"Example active ids (first 50): {active_ids[:50]}")

    # cleanup
    del sae, acts2d_all
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


After dropping non-binary is_refusal rows:
  coconot:    11473
  sorrybench: 440
  wildguard:  915
  xstest:     450

Counting active latents for layer 31 (HR + BC only)...
Total HR+BC examples across all datasets: 8111

Layer 31 SAE latents (HR+BC examples only):
  Total latents:              16384
  Latents ever active  (> 0): 760
  Latents always zero (== 0): 15624
